In [7]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time

In [8]:
from concurrent.futures import ThreadPoolExecutor, as_completed

# Set up Selenium WebDriver
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # Run without opening a browser
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

In [9]:
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

The purpose of this script is to webscrape the housing type based on postal codes to obtain a housing index. Subsequently, analysis is done to find correlation between housing index and disease prevalence. 

# Extracting from Active SG

In [4]:
base_url = "https://www.activesgcircle.gov.sg/facilities?page="

In [5]:
facilities = []
for page in range(1, 40):
    url = f"{base_url}{page}"
    #print(f"Fetching URL: {url}")
    driver.get(url)
    #print(driver.page_source) 
    time.sleep(5)  # Allow JavaScript to load content

    # Scroll to ensure lazy-loaded content appears
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(5)  # Wait for content to load

    # Find all facility divs
    facility_divs = driver.find_elements(By.CLASS_NAME, "cst-con-grp")
    print(f"Found {len(facility_divs)} facility divs on page {page}")

    if not facility_divs:
        print(f"⚠️ No facilities found on page {page}")

    for div in facility_divs:
        print(f"Facilities found on page {page}")
        try:
            name = div.find_element(By.TAG_NAME, "h4").text.strip()
            address = div.find_element(By.TAG_NAME, "h5").text.strip()
            facilities.append({"name": name, "address": address})
        except Exception as e:
            print(f"⚠️ Error extracting facility: {e}")

    print(f"✅ Scraped page {page}")

driver.quit()  # Close Selenium session

# Save data to CSV
df = pd.DataFrame(facilities)
df.to_csv("facilities.csv", index=False)

print(f"✅ Scraped {len(facilities)} facilities and saved to 'facilities.csv'.")


Found 10 facility divs on page 1
Facilities found on page 1
Facilities found on page 1
Facilities found on page 1
Facilities found on page 1
Facilities found on page 1
Facilities found on page 1
Facilities found on page 1
Facilities found on page 1
Facilities found on page 1
Facilities found on page 1
✅ Scraped page 1
Found 10 facility divs on page 2
Facilities found on page 2
Facilities found on page 2
Facilities found on page 2
Facilities found on page 2
Facilities found on page 2
Facilities found on page 2
Facilities found on page 2
Facilities found on page 2
Facilities found on page 2
Facilities found on page 2
✅ Scraped page 2
Found 10 facility divs on page 3
Facilities found on page 3
Facilities found on page 3
Facilities found on page 3
Facilities found on page 3
Facilities found on page 3
Facilities found on page 3
Facilities found on page 3
Facilities found on page 3
Facilities found on page 3
Facilities found on page 3
✅ Scraped page 3
Found 10 facility divs on page 4
Facilit

# Extracting community centres postal codes from gov.sg

#### Getting the list of links
Link of list is saved in a page, in an element called ministries, tag name 'a'

In [5]:
# Get the list of links

# Initialize WebDriver (Make sure to use the correct WebDriver for your browser)
driver = webdriver.Chrome()  

# Open the webpage
url = "https://www.sgdi.gov.sg/other-organisations/community-centres"
driver.get(url)

# Find the div with class "directory-links"
directory_div = driver.find_element(By.CLASS_NAME, "ministries")

# Find all <a> tags inside the div
links = directory_div.find_elements(By.TAG_NAME, "a")
link_list = [link.get_attribute("href") for link in links if link.get_attribute("href")]

print(link_list)

['https://www.sgdi.gov.sg/ministries/mccy/statutory-boards/pa/departments/grassroots/departments/cc/departments/acetp', 'https://www.sgdi.gov.sg/ministries/mccy/statutory-boards/pa/departments/grassroots/departments/cc/departments/ajcc', 'https://www.sgdi.gov.sg/ministries/mccy/statutory-boards/pa/departments/grassroots/departments/cc/departments/avcc', 'https://www.sgdi.gov.sg/ministries/mccy/statutory-boards/pa/departments/grassroots/departments/cc/departments/amkcc', 'https://www.sgdi.gov.sg/ministries/mccy/statutory-boards/pa/departments/grassroots/departments/cc/departments/arcc', 'https://www.sgdi.gov.sg/ministries/mccy/statutory-boards/pa/departments/grassroots/departments/cc/departments/bcc', 'https://www.sgdi.gov.sg/ministries/mccy/statutory-boards/pa/departments/grassroots/departments/cc/departments/bcc-copy', 'https://www.sgdi.gov.sg/ministries/mccy/statutory-boards/pa/departments/grassroots/departments/cc/departments/bncc', 'https://www.sgdi.gov.sg/ministries/mccy/statutory

In [12]:
# Comb through all the websites
driver = webdriver.Chrome()  
facilities = []
for onelink in link_list:
    url = onelink
    #print(f"Fetching URL: {url}")
    driver.get(url)
    #print(driver.page_source) 
    try:
        div = driver.find_element(By.CLASS_NAME, "agency")
        name = div.find_element(By.TAG_NAME, "h1").text.strip()
        address = driver.find_element(By.CLASS_NAME, "street-address").text.strip()
        facilities.append({"name": name, "address": address})
    except Exception as e:
        print(f"⚠️ Error extracting facility: {e}")

driver.quit()  # Close Selenium session

# Save data to CSV
df = pd.DataFrame(facilities)
df.to_csv("CCRC.csv", index=False)

print(f"✅ Scraped {len(facilities)} facilities and saved to 'facilities.csv'.")

# Close the driver
driver.quit()

⚠️ Error extracting facility: Message: no such element: Unable to locate element: {"method":"css selector","selector":".street-address"}
  (Session info: chrome=132.0.6834.160); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#no-such-element-exception
Stacktrace:
	GetHandleVerifier [0x00007FF6D40F02F5+28725]
	(No symbol) [0x00007FF6D4052AE0]
	(No symbol) [0x00007FF6D3EE510A]
	(No symbol) [0x00007FF6D3F393D2]
	(No symbol) [0x00007FF6D3F395FC]
	(No symbol) [0x00007FF6D3F83407]
	(No symbol) [0x00007FF6D3F5FFEF]
	(No symbol) [0x00007FF6D3F80181]
	(No symbol) [0x00007FF6D3F5FD53]
	(No symbol) [0x00007FF6D3F2A0E3]
	(No symbol) [0x00007FF6D3F2B471]
	GetHandleVerifier [0x00007FF6D441F30D+3366989]
	GetHandleVerifier [0x00007FF6D44312F0+3440688]
	GetHandleVerifier [0x00007FF6D44278FD+3401277]
	GetHandleVerifier [0x00007FF6D41BAAAB+858091]
	(No symbol) [0x00007FF6D405E74F]
	(No symbol) [0x00007FF6D405A304]
	(No symbol) [0x0000

In [11]:
facilities


[{'name': 'Singapore Government Directory',
  'address': 'Block 547 Woodlands Drive 16 #01-177\nSingapore 730547'},
 {'name': 'Singapore Government Directory',
  'address': 'Block 110 Hougang Avenue 1 #01-1048\nSingapore 530110'},
 {'name': 'Singapore Government Directory',
  'address': '59 Anchorvale Road\n \nSingapore 544965'},
 {'name': 'Singapore Government Directory',
  'address': '795 Ang Mo Kio Avenue 1\n Singapore 569976'},
 {'name': 'Singapore Government Directory',
  'address': '150 Pandan Gardens\nSingapore 609335'},
 {'name': 'Singapore Government Directory',
  'address': '850 New Upper Changi Road\n Singapore 467352'},
 {'name': 'Singapore Government Directory',
  'address': '51 Bishan Street 13\nSingapore 579799'},
 {'name': 'Singapore Government Directory',
  'address': 'Block 233 Bishan Street 22 #01-126\nSingapore 570233'},
 {'name': 'Singapore Government Directory',
  'address': '10 Boon Lay Place\nSingapore 649882'},
 {'name': 'Singapore Government Directory',
  'add

# CondoSG

- there are 4 types of private housing listed in this website.
- As the number of condos are large, multithreading is used to speed up the extraction. 
- condo names and addresses can be extracted, however there are no postal codes in this website, hence we have to map these addresses and building names postal codes using OneMap's api call.

In [20]:
# Base URL with placeholder
base_url = "https://condo.singaporeexpats.com/{}/property/condo"

# Function to scroll and wait for lazy-loaded elements
def scroll_and_wait(driver):
    last_height = driver.execute_script("return document.body.scrollHeight")
    
    while True:
        # Scroll down to the bottom
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)  # Allow time for new elements to load
        
        # Check new page height
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:  
            break  # Stop scrolling if no new content loads
        last_height = new_height
        
# Function to scrape a single page
def scrape_page(page):

    driver = webdriver.Chrome() 

    url = base_url.format(page)
    driver.get(url)
    
    facilities = []

    scroll_and_wait(driver)

    try:
        listdivs = driver.find_elements(By.CLASS_NAME, "listdiv")
        
        for div in listdivs:
            try:
                title = div.find_element(By.CLASS_NAME,"title").text.strip()
                name = div.find_element(By.CLASS_NAME, "listcol1").text.strip()
                desc = div.find_element(By.CLASS_NAME, "listcol2").text.strip()
                facilities.append({"title": title, "name": name, "desc": desc})
                
            except Exception as e:
                print(f"⚠️ Error extracting facility on page {page}: {e}")
        print(page, len(facilities))
    except Exception as e:
        print(f"⚠️ Error on page {page}: {e}")
    print(url)
    driver.quit()
    return facilities

# Start multithreading
start_time = time.time()
all_facilities = []

with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(scrape_page, page): page for page in range(1, 53)}
    
    for future in as_completed(futures):
        all_facilities.extend(future.result())

end_time = time.time()
print(f"✅ Scraped {len(all_facilities)} facilities in {end_time - start_time:.2f} seconds")
print(all_facilities)


2 50
https://condo.singaporeexpats.com/2/property/condo
10 50
https://condo.singaporeexpats.com/10/property/condo
3 50
https://condo.singaporeexpats.com/3/property/condo
4 50
https://condo.singaporeexpats.com/4/property/condo
9 50
https://condo.singaporeexpats.com/9/property/condo
7 50
https://condo.singaporeexpats.com/7/property/condo
8 50
https://condo.singaporeexpats.com/8/property/condo
5 50
https://condo.singaporeexpats.com/5/property/condo
6 50
https://condo.singaporeexpats.com/6/property/condo
1 50
https://condo.singaporeexpats.com/1/property/condo
11 50
https://condo.singaporeexpats.com/11/property/condo
12 50
https://condo.singaporeexpats.com/12/property/condo
13 50
https://condo.singaporeexpats.com/13/property/condo
15 50
https://condo.singaporeexpats.com/15/property/condo
18 50
https://condo.singaporeexpats.com/18/property/condo
16 50
https://condo.singaporeexpats.com/16/property/condo
17 50
https://condo.singaporeexpats.com/17/property/condo
19 50
https://condo.singaporeexp

In [21]:
len(all_facilities)

2552

In [22]:
df = pd.DataFrame(all_facilities)
df.to_csv("Condo2.csv", index=False)

In [13]:
driver = webdriver.Chrome() 
#print(f"Fetching URL: {url}")


url = "https://condo.singaporeexpats.com/property/service-apartment"  # Replace with your target URL
facilities = []

driver.get(url)
#print(driver.page_source) 

listdivs = driver.find_elements(By.CLASS_NAME, "listdiv")

# Print the extracted elements
for idx, div in enumerate(listdivs, start=1):
    try:
        title = div.find_element(By.CLASS_NAME,"title").text.strip()
        name = div.find_element(By.CLASS_NAME, "listcol1").text.strip()
        desc = driver.find_element(By.CLASS_NAME, "listcol2").text.strip()
        facilities.append({"title": title, "name": name, "desc": desc})
    except Exception as e:
        print(f"⚠️ Error extracting facility: {e}")

driver.quit()  # Close Selenium session


In [14]:
df = pd.DataFrame(facilities)
df.to_csv("ServiceApartment.csv", index=False)

In [15]:
driver = webdriver.Chrome() 
#print(f"Fetching URL: {url}")

base_url = "https://condo.singaporeexpats.com/{}/property/cluster-housing"  # Replace with your target URL
facilities = []

for page in range(1,4):
    url = base_url.format(page)
    driver.get(url)
    #print(driver.page_source) 
    
    listdivs = driver.find_elements(By.CLASS_NAME, "listdiv")
    
    # Print the extracted elements
    for idx, div in enumerate(listdivs, start=1):
        try:
            title = div.find_element(By.CLASS_NAME,"title").text.strip()
            name = div.find_element(By.CLASS_NAME, "listcol1").text.strip()
            desc = driver.find_element(By.CLASS_NAME, "listcol2").text.strip()
            facilities.append({"title": title, "name": name, "desc": desc})
        except Exception as e:
            print(f"⚠️ Error extracting facility: {e}")

driver.quit()  # Close Selenium session


In [16]:
df = pd.DataFrame(facilities)
df.to_csv("ClusterHousing.csv", index=False)

In [17]:
# Base URL with placeholder
base_url = "https://condo.singaporeexpats.com/{}/property/landed-estate"

# Function to scrape a single page
def scrape_page(page):
    #options = webdriver.ChromeOptions()
    #options.add_argument("--headless")  # Run in headless mode
    driver = webdriver.Chrome() 

    url = base_url.format(page)
    print(url)
    driver.get(url)
    facilities = []

    try:
        listdivs = driver.find_elements(By.CLASS_NAME, "listdiv")
        
        for div in listdivs:
            try:
                title = div.find_element(By.CLASS_NAME,"title").text.strip()                
                name = div.find_element(By.CLASS_NAME, "listcol1").text.strip()
                desc = div.find_element(By.CLASS_NAME, "listcol2").text.strip()
                facilities.append({"title": title, "name": name, "desc": desc})
            except Exception as e:
                print(f"⚠️ Error extracting facility on page {page}: {e}")

    except Exception as e:
        print(f"⚠️ Error on page {page}: {e}")

    driver.quit()
    return facilities

# Start multithreading
start_time = time.time()
all_facilities = []

with ThreadPoolExecutor(max_workers=4) as executor:  # 4 threads
    futures = {executor.submit(scrape_page, page): page for page in range(1, 9)}
    
    for future in as_completed(futures):
        all_facilities.extend(future.result())

end_time = time.time()
print(f"✅ Scraped {len(all_facilities)} facilities in {end_time - start_time:.2f} seconds")



https://condo.singaporeexpats.com/2/property/landed-estate
https://condo.singaporeexpats.com/1/property/landed-estate
https://condo.singaporeexpats.com/3/property/landed-estate
https://condo.singaporeexpats.com/4/property/landed-estate
https://condo.singaporeexpats.com/5/property/landed-estate
https://condo.singaporeexpats.com/6/property/landed-estate
https://condo.singaporeexpats.com/7/property/landed-estate
https://condo.singaporeexpats.com/8/property/landed-estate
✅ Scraped 349 facilities in 85.33 seconds


In [18]:
df = pd.DataFrame(all_facilities)
df.to_csv("landedEstate.csv", index=False)

# One map
- As mentioned above, the public housing data collected from CondoSG does not have postal code, we search Onemap based on building name and address to obtain possible postal codes.
- Addresses obtained from CondoSG contain ranges of block numbers (e.g. block numbers are 1-17 or 2,4,6). These entries are split such that each block and address has a line for the search 
- Subsequently, we clean the data by ensuring that the building name collected is the full building name. 

#### Helper functions

In [10]:
import re
import requests
import pandas as pd

def split_columns(df):
    df[['Type', 'Address','Developer']] = df['name'].str.split('\n', expand=True)
    df[['District', 'Units','Tenure','Estimated TOP']] = df['desc'].str.split('\n', expand=True)
    df['Type'] = df['Type'].str.replace('Type: ','', regex=False)
    df['Address'] = df['Address'].str.replace('Address: ','', regex=False)
    df['Developer'] = df['Developer'].str.replace('Developer: ','', regex=False)
    df['District'] = df['District'].str.replace('District: ','', regex=False)
    df['Units'] = df['Units'].str.replace('Units: ','', regex=False)
    df['Estimated TOP'] = df['Estimated TOP'].str.replace('Estimated TOP: ','', regex=False)
    return df

def extract_blocks_and_addresses(df):
    """Expand the DataFrame so each row has only one block number."""
    expanded_rows = []
    
    for _, row in df.iterrows():
        address = row['Address']
        
        # Extract street name by removing block numbers, ranges, and commas
        street_name = re.sub(r"^\d+[A-Z]?(\s*[-,]\s*\d+[A-Z]?)*\s*", "", address).strip()
        
        # Extract comma-separated blocks
        if "," in address:
            blocks = re.findall(r"\d+", address)  # Extract all numbers
        else:
            # Extract numeric range or single block number
            match_range = re.match(r"(\d+)(?:\s*-\s*(\d+))?", address)
            if match_range:
                start, end = match_range.groups()
                if end:  # If a range exists, expand it
                    blocks = list(map(str, range(int(start), int(end) + 1)))
                else:
                    blocks = [start]
            else:
                blocks = [address]  # If no blocks found, keep original address
        
        # Create new rows with expanded block numbers
        for block in blocks:
            new_row = row.copy()
            new_row['Address'] = f"{block} {street_name}"
            expanded_rows.append(new_row)
    
    return pd.DataFrame(expanded_rows)


# API request function with retry logic
def fetch_address(postal, type, retries=3, delay=2):
    base_url = "https://www.onemap.gov.sg/api/common/elastic/search?searchVal={}&returnGeom=Y&getAddrDetails=Y&pageNum={}"
    columns = ["SEARCHVAL", "BLK_NO", "ROAD_NAME", "BUILDING", "ADDRESS", "X", "Y", "LATITUDE", "LONGITUDE", "search_term"]
    results_list = [] 
    
    for attempt in range(1, retries + 1):
        try:
            url = base_url.format(postal, 1)
            response = requests.get(url, timeout=10)
            if response.status_code == 200:
                data = response.json()
                totpage = data["totalNumPages"]                

                if type == "address": 
                    if totpage >= 2:
                        return []  # Ignore if too many results
                    if "results" in data and len(data["results"]) > 0:
                        entry = data["results"][0]
                        results_list.append({
                            "SEARCHVAL": entry.get("SEARCHVAL", ""),
                            "BLK_NO": entry.get("BLK_NO", ""),
                            "ROAD_NAME": entry.get("ROAD_NAME", ""),
                            "BUILDING": entry.get("BUILDING", ""),
                            "ADDRESS": entry.get("ADDRESS", ""),
                            "X": entry.get("X", ""),
                            "Y": entry.get("Y", ""),
                            "LATITUDE": entry.get("LATITUDE", ""),
                            "LONGITUDE": entry.get("LONGITUDE", ""),
                            "search_term": postal
                            
                        })
                        return results_list 

                if type == "building":
                    if totpage >= 1000:
                        return []  # Ignore if too many results
                    for page in range(1, totpage + 1):
                        url = base_url.format(postal, page)
                        response = requests.get(url, timeout=10)
                        data = response.json()
                        if "results" in data and len(data["results"]) > 0:
                            for entry in data["results"]:
                                results_list.append({
                                    "SEARCHVAL": entry.get("SEARCHVAL", ""),
                                    "BLK_NO": entry.get("BLK_NO", ""),
                                    "ROAD_NAME": entry.get("ROAD_NAME", ""),
                                    "BUILDING": entry.get("BUILDING", ""),
                                    "ADDRESS": entry.get("ADDRESS", ""),
                                    "X": entry.get("X", ""),
                                    "Y": entry.get("Y", ""),
                                    "LATITUDE": entry.get("LATITUDE", ""),
                                    "LONGITUDE": entry.get("LONGITUDE", ""),
                                    "search_term": postal
                                })
                    return results_list
                    
        except requests.exceptions.RequestException as e:
            print(f"❌ Error: {e}. Retrying {attempt}/{retries}...")
            time.sleep(delay * attempt)
            
    print(f"❌ Failed after {retries} retries: {postal}")
    return []

#### Condo

In [11]:
Condo = pd.read_csv('Condo2.csv')
Condo = split_columns(Condo)
Condo = extract_blocks_and_addresses(Condo)
Condo.to_csv('Condo_cleaned.csv')

# Building_list = Condo['title'].to_list()
Address_list = Condo['Address'].to_list()

# Start multithreading
start_time = time.time()
all_facilities = []

with ThreadPoolExecutor(max_workers=4) as executor:  # 4 threads
    futures = {executor.submit(fetch_address, search_val, "building") for search_val in Building_list}
    
    for future in as_completed(futures):
        all_facilities.extend(future.result())

end_time = time.time()

print(len(all_facilities))

df = pd.DataFrame(all_facilities)
#df.to_csv("Onemap_condo_by_address.csv", index=False)
df.to_csv("Onemap_Condo_by_buildingName.csv", index=False)

# Start multithreading
start_time = time.time()
all_facilities = []

with ThreadPoolExecutor(max_workers=4) as executor:  # 4 threads
    futures = {executor.submit(fetch_address, search_val, "address") for search_val in Address_list}
    
    for future in as_completed(futures):
        all_facilities.extend(future.result())

end_time = time.time()

print(len(all_facilities))

df = pd.DataFrame(all_facilities)

df.to_csv("Onemap_Condo_by_address.csv", index=False)

❌ Error: HTTPSConnectionPool(host='www.onemap.gov.sg', port=443): Read timed out. (read timeout=10). Retrying 1/3...❌ Error: HTTPSConnectionPool(host='www.onemap.gov.sg', port=443): Read timed out. (read timeout=10). Retrying 1/3...

❌ Error: HTTPSConnectionPool(host='www.onemap.gov.sg', port=443): Read timed out. (read timeout=10). Retrying 1/3...
8004


#### Cluster Housing

In [32]:
# Define search list
ClusterHousing = pd.read_csv('ClusterHousing.csv')
ClusterHousing = split_columns(ClusterHousing)
ClusterHousing = extract_blocks_and_addresses(ClusterHousing)
ClusterHousing.to_csv('ClusterHousing_cleaned.csv')

#Building_list = ClusterHousing['title'].to_list()
Address_list = ClusterHousing['Address'].to_list()

# Start multithreading
start_time = time.time()
all_facilities = []

with ThreadPoolExecutor(max_workers=4) as executor:  # 4 threads
    futures = {executor.submit(fetch_address, search_val, "building") for search_val in Building_list}
    
    for future in as_completed(futures):
        all_facilities.extend(future.result())

end_time = time.time()

print(len(all_facilities))

df = pd.DataFrame(all_facilities)
#df.to_csv("Onemap_condo_by_address.csv", index=False)
df.to_csv("Onemap_ClusterHousing_by_buildingName.csv", index=False)

# Start multithreading
start_time = time.time()
all_facilities = []

with ThreadPoolExecutor(max_workers=4) as executor:  # 4 threads
    futures = {executor.submit(fetch_address, search_val, "address") for search_val in Address_list}
    
    for future in as_completed(futures):
        all_facilities.extend(future.result())

end_time = time.time()

print(len(all_facilities))

df = pd.DataFrame(all_facilities)
#df.to_csv("Onemap_condo_by_address.csv", index=False)
df.to_csv("Onemap_ClusterHousing_by_address.csv", index=False)

2804


#### Landed estate

In [33]:
# Define search list
landedEstate = pd.read_csv('landedEstate.csv')
landedEstate = split_columns(landedEstate)
landedEstate = extract_blocks_and_addresses(landedEstate)
landedEstate.to_csv('landedEstate_cleaned.csv')

# Building_list = landedEstate['title'].to_list()
Address_list = landedEstate['Address'].to_list()

# Start multithreading
start_time = time.time()
all_facilities = []

with ThreadPoolExecutor(max_workers=4) as executor:  # 4 threads
    futures = {executor.submit(fetch_address, search_val, "building") for search_val in Building_list}
    
    for future in as_completed(futures):
        all_facilities.extend(future.result())

end_time = time.time()
print('time taken to search by building name =',end_time-start_time)

print(len(all_facilities))

df = pd.DataFrame(all_facilities)
df.to_csv("Onemap_landedEstate_by_buildingName.csv", index=False)

# Start multithreading
start_time = time.time()
all_facilities = []

with ThreadPoolExecutor(max_workers=4) as executor:  # 4 threads
    futures = {executor.submit(fetch_address, search_val, "address") for search_val in Address_list}
    
    for future in as_completed(futures):
        all_facilities.extend(future.result())

end_time = time.time()

print(len(all_facilities))

df = pd.DataFrame(all_facilities)
df.to_csv("Onemap_landedEstate_by_address.csv", index=False)

❌ Error: HTTPSConnectionPool(host='www.onemap.gov.sg', port=443): Read timed out. (read timeout=10). Retrying 1/3...
❌ Error: HTTPSConnectionPool(host='www.onemap.gov.sg', port=443): Read timed out. (read timeout=10). Retrying 1/3...
4538


#### Service apartment

In [34]:
# Define search list
ServiceApartment = pd.read_csv('ServiceApartment.csv')
ServiceApartment = split_columns(ServiceApartment)
ServiceApartment = extract_blocks_and_addresses(ServiceApartment)
ServiceApartment.to_csv('ServiceApartment_cleaned.csv')

# Building_list = ServiceApartment['title'].to_list()
Address_list = ServiceApartment['Address'].to_list()

#Start multithreading
start_time = time.time()
all_facilities = []

with ThreadPoolExecutor(max_workers=4) as executor:  # 4 threads
    futures = {executor.submit(fetch_address, search_val, "building") for search_val in Building_list}
    
    for future in as_completed(futures):
        all_facilities.extend(future.result())

end_time = time.time()

print(len(all_facilities))

df = pd.DataFrame(all_facilities)
#df.to_csv("Onemap_condo_by_address.csv", index=False)
df.to_csv("Onemap_ServiceApartment_by_buildingName.csv", index=False)

# Start multithreading
start_time = time.time()
all_facilities = []

with ThreadPoolExecutor(max_workers=4) as executor:  # 4 threads
    futures = {executor.submit(fetch_address, search_val, "address") for search_val in Address_list}
    
    for future in as_completed(futures):
        all_facilities.extend(future.result())

end_time = time.time()

print(len(all_facilities))

df = pd.DataFrame(all_facilities)
#df.to_csv("Onemap_condo_by_address.csv", index=False)
df.to_csv("Onemap_ServiceApartment_by_address.csv", index=False)

46
